# 주택 가격 데이터를 활용한 회귀 모델 학습 및 예측

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

#1. 데이터셋 불러오기
df = pd.read_csv('train.csv')

#2. LotFrontage는 평균으로 대체
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].mean())

#2-1. 결측값 처리 - NaN
search_null = df.isnull().sum()
null_ratio = search_null / len(df)
cols_to_drop = null_ratio[null_ratio > 0.15].index
df = df.drop(columns=cols_to_drop)

#2-3. 남은 결측치 마저 처리
for col in df.columns:
    #만약 열에 null이 0개 초과했을 때, 그리고 값이 정수, 실수일때,
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            #평균값으로 채울 것
            df[col] = df[col].fillna(df[col].mean())
        else:
            #최빈값으로 채울 것
            df[col] = df[col].fillna(df[col].mode()[0])
            
#3. 불필요한 열 제거: Id
df = df.drop(columns=['Id'])

#4. 범주형 데이터 인코딩
df = pd.get_dummies(df)

#4-1. X, y 분리
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

#5. 학습/테스트 데이터 분리: 8:2 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#6. 모델 학습
model = DecisionTreeRegressor(random_state=42)
train_model = model.fit(X_train, y_train)

#6-1. 예측
y_pred = model.predict(X_test)

#7. 모델 평가
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

#8. 최종 결과 도출
print(f"MAE  (평균 절대 오차): {mae:,.2f}")
print(f"MSE  (평균 제곱 오차): {mse:,.2f}")
print(f"RMSE (평균 제곱근 오차): {rmse:,.2f}")
print(f"R2   (결정계수): {r2:.4f}")
print(f"\n해석: 예측값이 실제 주택 가격과 평균적으로 약 {mae:,.0f}달러 정도 차이가 나며,")
print(f"모델이 SalePrice 분산의 약 {r2*100:.1f}%를 설명하고 있습니다.")

MAE  (평균 절대 오차): 28,428.47
MSE  (평균 제곱 오차): 1,871,362,744.31
RMSE (평균 제곱근 오차): 43,259.25
R2   (결정계수): 0.7560

해석: 예측값이 실제 주택 가격과 평균적으로 약 28,428달러 정도 차이가 나며,
모델이 SalePrice 분산의 약 75.6%를 설명하고 있습니다.
